<a href="https://colab.research.google.com/github/daliaray/Daliaray/blob/main/PREDICTWEEK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Load the CSV data
data = pd.read_csv('attendance_data.csv')

# Define the sequence length (number of past weeks to consider for prediction)
sequence_length = 4

# Convert 'semaine' (green, red, normal) into numerical labels
label_encoder = LabelEncoder()
data['semaine_encoded'] = label_encoder.fit_transform(data['semaine'])

# Convert 'status' to numerical values (1 for present, 0 for absent)
status_map = {'present': 1, 'absent': 0}
data['status_numeric'] = data['status'].map(status_map)

# Prepare sequences and labels
X = []
y = []

for student_id, student_data in data.groupby('student_id'):
    student_data = student_data.sort_values('date')
    # Ensure there's enough data for at least one sequence
    if len(student_data) > sequence_length:
        for i in range(len(student_data) - sequence_length):
            # Use the numeric 'status_numeric' column for X
            X.append(student_data['status_numeric'].values[i:i+sequence_length])
            y.append(student_data['semaine_encoded'].values[i+sequence_length])

X = np.array(X)
y = np.array(y)

# Reshape X for LSTM input (samples, time_steps, features)
X = X.reshape((X.shape[0], X.shape[1], 1))

# Split the data into training, validation, and test sets
# Adjusted test_size from 0.2 to 0.5 to ensure X_temp has at least 2 samples for the subsequent split.
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.5, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Create an LSTM model
model = Sequential()
model.add(LSTM(units=50, input_shape=(sequence_length, 1)))
model.add(Dense(3, activation='softmax'))  # 3 classes for weeks (green, red, normal)

# Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val))

# Evaluate the model on the test set
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}')

# Use the trained model for predictions (e.g., predicting future absenteeism categories)
future_sequence = np.array([[0, 1, 0, 1]])  # Replace with your own future absenteeism data (0 for absent, 1 for present)
# Reshape future_sequence for the model
future_sequence = future_sequence.reshape((1, sequence_length, 1))

# model.predict_classes is deprecated, using model.predict and argmax
predictions = model.predict(future_sequence)
predicted_label = np.argmax(predictions, axis=1)
predicted_category = label_encoder.inverse_transform(predicted_label)
print(f'Predicted Category for Future Week: {predicted_category[0]}')

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5000 - loss: 1.0694 - val_accuracy: 0.0000e+00 - val_loss: 1.1786
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.5000 - loss: 1.0589 - val_accuracy: 0.0000e+00 - val_loss: 1.1983
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.5000 - loss: 1.0486 - val_accuracy: 0.0000e+00 - val_loss: 1.2182
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.5000 - loss: 1.0385 - val_accuracy: 0.0000e+00 - val_loss: 1.2383
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - accuracy: 0.5000 - loss: 1.0286 - val_accuracy: 0.0000e+00 - val_loss: 1.2586
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.5000 - loss: 1.0189 - val_accuracy: 0.0000e+00 - val_loss: 1.2792
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.5000 - loss: 1.0094 - val_accuracy: 0.0000e+00 - val_loss: 1.3002
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.5000 - loss: 1.0001 - val_accuracy: 0.0000e

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
Predicted Category for Future Week: green
